In [2]:
import sys
!{sys.executable} -m pip install opencv-python

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/40.2 MB ? eta -:--:--
   ---------------------------------------- 0.3/40.2 MB ? eta -:--:--
    --------------------------------------- 0.8/40.2 MB 3.2 MB/s eta 0:00:13
   - -------------------------------------- 1.3/40.2 MB 3.3 MB/s eta 0:00:12
   - -------------------------------------- 1.6/40.2 MB 2.6 MB/s eta 0:00:15
   - -------------------------------------- 1.8/40.2 MB 2.1 MB/s eta 0:00:19
   -- ------------------------------------- 2.1/40.2 MB 2.0 MB/s eta 0:00:20
   -- ------------------------------------- 2.4/40.2 MB 1.7 MB/s eta 0:00:22
   -- ------------------------------------- 2.6/40.2 MB 1.6 MB/s eta 0:00:23
   -- ------------------------------------- 2.9/40.2 MB 1.6 MB/s eta 0:00:24
   --- ------------------------------------ 3.1/40.2 MB 1.5 MB/s eta 0:00:24
   --- ------------------------------------ 3.4/40.2 MB 1.5 MB/s eta 0:00:25
   --- -----

In [3]:
import cv2
print(f"OpenCV version: {cv2.__version__}")

OpenCV version: 4.13.0


In [5]:
import sys
!{sys.executable} -m pip install ultralytics

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------- ----- 1.0/1.2 MB 9.8 MB/s eta 0:00:01
   ---------------------------------- ----- 1.0/1.2 MB 9.8 MB/s eta 0:00:01
   ---------------------------------------- 1.2/1.2 MB 2.0 MB/s  0:00:00
   ---------------------------------------- 0.0/824.0 kB ? eta -:--:--
   ------------ --------------------------- 262.1/824.0 kB ? eta -:--:--
   ------------------------ ------------- 524.3/824.0 kB 976.4 kB/s eta 0:00:01
   ------------------------ ------------- 524.3/824.0 kB 976.4 kB/s eta 0:00:01
   ------------------------ ------------- 524.3/824.0 kB 976.4 kB/s eta 0:00:01
   ---------------------------------------- 824.0/824.0 kB 675.7 kB/s  0:00:00
   ---------------------------------------- 0.0/47.0 MB ? eta -:--:--
   ---------------------------------------- 0.5/47.0 MB 4.2 MB/s eta 0:00:11
   ---------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [6]:
import cv2
import pandas as pd
import os
from ultralytics import YOLO
from datetime import datetime

# 1. INITIALIZE THE AI MODEL
# 'yolov8n.pt' is the 'Nano' version—it's very fast for real-time video.
# The first time you run this, it will download the model file (~6MB).
model = YOLO('yolov8n.pt') 

# 2. CONFIGURATION
VIDEO_SOURCE = 0  # Use 0 for Webcam. Or use a string path: 'site_video.mp4'
OUTPUT_FILE = "safety_compliance_report.csv"
violation_records = []
logged_ids = set() # To ensure we only log a person once

def run_safety_project():
    # Initialize video capture
    cap = cv2.VideoCapture(VIDEO_SOURCE)
    
    if not cap.isOpened():
        print("Error: Could not open video source.")
        return

    print("AI Safety Monitor Starting... Press 'q' to stop.")

    while True:
        success, frame = cap.read()
        if not success:
            break

        # 3. AI DETECTION & TRACKING
        # persist=True allows the AI to give each person a unique ID number
        results = model.track(frame, persist=True, conf=0.5, verbose=False)

        # 4. PROCESS DETECTIONS
        if results[0].boxes.id is not None:
            boxes = results[0].boxes.xyxy.int().cpu().tolist()
            track_ids = results[0].boxes.id.int().cpu().tolist()
            class_ids = results[0].boxes.cls.int().cpu().tolist()

            for box, track_id, class_id in zip(boxes, track_ids, class_ids):
                label = model.names[class_id]
                
                # We are looking for 'person' class in the base model
                if label == 'person':
                    x1, y1, x2, y2 = box
                    
                    # --- SIMULATED LOGIC FOR THE REAL WORLD ---
                    # In a custom PPE model, you would check if class is 'No-Helmet'
                    # Here, we will simulate a violation for Demo purposes:
                    is_violating = (track_id % 3 == 0) # Simulate every 3rd person as a violation
                    
                    if is_violating:
                        color = (0, 0, 255) # Red for danger
                        status = "VIOLATION: No PPE"
                        
                        # Log to CSV if not already logged
                        if track_id not in logged_ids:
                            timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                            violation_records.append({"Worker_ID": track_id, "Time": timestamp, "Status": "No Helmet"})
                            logged_ids.add(track_id)
                    else:
                        color = (0, 255, 0) # Green for safe
                        status = "COMPLIANT"

                    # 5. DRAW CUSTOM UI
                    cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                    cv2.putText(frame, f"ID:{track_id} {status}", (x1, y1 - 10),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

        # 6. DASHBOARD OVERLAY
        cv2.rectangle(frame, (0, 0), (300, 60), (0, 0, 0), -1)
        cv2.putText(frame, f"Active Violations: {len(violation_records)}", (10, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

        # Show the video window
        cv2.imshow("Smart Site Safety Monitor", frame)

        # Exit on 'q' key
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    # 7. CLEANUP & SAVE DATA
    cap.release()
    cv2.destroyAllWindows()
    
    if violation_records:
        df = pd.DataFrame(violation_records)
        df.to_csv(OUTPUT_FILE, index=False)
        print(f"--- SESSION FINISHED ---")
        print(f"Report saved as: {OUTPUT_FILE}")
        print(df.head())
    else:
        print("Session finished. No violations detected.")

# Run the program
if __name__ == "__main__":
    run_safety_project()

Creating new Ultralytics Settings v0.0.6 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\Hp\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
AI Safety Monitor Starting... Press 'q' to stop.
requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------- ----------- 1.0/1.5 MB 8.2 MB/s eta 0:00:01
   ---------------------------- ----------- 1.0/1.5 MB 8.2 MB/s eta 0:00:01
   ---------------------------- ----------- 1.0/1.5 MB 8.2 MB/s eta 0:00:01
   ----------------------------------- ---- 1.3/1.5 MB 1.5 MB/s eta 0:00:01
   --------------